# Run xlangai/spider test questions against `/ask`

This notebook loads the `xlangai/spider` dataset from Hugging Face, takes every row in the `test` split, sends a POST request to `http://localhost:8081/ask`, appends the response to the original dataframe, and saves the enriched results as JSON.

The request body uses a fixed template and replaces:

- `question` with the dataset row's `question`
- `schema_name` with the dataset row's database/schema identifier, usually `db_id`


## 1. Install and import dependencies

In [23]:
# Uncomment this if you are running in a fresh environment.
# %pip install -q datasets pandas requests tqdm

In [24]:
import json
import time
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd
import requests
from datasets import load_dataset
from tqdm.auto import tqdm

## 2. Configuration

In [25]:
DATASET_NAME = "xlangai/spider"
SPLIT = "validation" # validation, train

ASK_URL = "http://localhost:8081/ask"

REQUEST_TIMEOUT_SECONDS = 120
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.0

# Set to an integer like 10 for a smoke test, or None to process the whole split.
MAX_ROWS: Optional[int] = 234
OFFSET: Optional[int] = 800

max_rows_part = MAX_ROWS if MAX_ROWS is not None else "all"
OUTPUT_PATH = Path(f"result/spider_test_with_ask_responses_offset_{OFFSET}_max_rows_{max_rows_part}.json")

BASE_PAYLOAD: Dict[str, Any] = {
    "user_id": "user_123",
    "model": "google/gemma-4-E4B-it",
    "first_user_message": "I want to ask a question about my database schema.",
    "question": None,
    "is_thinking": None,
    "skip_user_interaction": True,
    "schema_name": None,
}

## 3. Load the Spider test split from Hugging Face

In [26]:
dataset = load_dataset(DATASET_NAME, split=SPLIT)
df = dataset.to_pandas()

print(f"Loaded {len(df):,} rows from {DATASET_NAME}/{SPLIT}")
print("Columns:", list(df.columns))
df.head()

Loaded 1,034 rows from xlangai/spider/validation
Columns: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks']


,db_id,query,question,query_toks,query_toks_no_value,question_toks
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[How, many, singers, do, we, have, ?]"
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[What, is, the, total, number, of, singers, ?]"
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","Show name, country, age for all singers ordere...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[Show, name, ,, country, ,, age, for, all, sin..."
3,concert_singer,"SELECT name , country , age FROM singer ORDE...","What are the names, countries, and ages for ev...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[What, are, the, names, ,, countries, ,, and, ..."
4,concert_singer,"SELECT avg(age) , min(age) , max(age) FROM s...","What is the average, minimum, and maximum age ...","[SELECT, avg, (, age, ), ,, min, (, age, ), ,,...","[select, avg, (, age, ), ,, min, (, age, ), ,,...","[What, is, the, average, ,, minimum, ,, and, m..."


## 4. Validate required columns

Spider-style datasets usually store the natural-language question in `question` and the database/schema identifier in `db_id`. This cell fails early if those fields are not present.

In [27]:
QUESTION_COLUMN = "question"
SCHEMA_COLUMN_CANDIDATES = ["db_id", "schema_name", "database_id"]

missing = []
if QUESTION_COLUMN not in df.columns:
    missing.append(QUESTION_COLUMN)

schema_column = next((col for col in SCHEMA_COLUMN_CANDIDATES if col in df.columns), None)
if schema_column is None:
    missing.append("one of: " + ", ".join(SCHEMA_COLUMN_CANDIDATES))

if missing:
    raise ValueError(f"Missing required dataset column(s): {missing}. Available columns: {list(df.columns)}")

print(f"Using question column: {QUESTION_COLUMN}")
print(f"Using schema column: {schema_column}")

Using question column: question
Using schema column: db_id


## 5. Define request helper

In [28]:
def build_payload(row: pd.Series, thinking: bool) -> Dict[str, Any]:
    """Create the /ask request body for one Spider row."""
    payload = dict(BASE_PAYLOAD)
    payload["is_thinking"] = thinking
    payload["question"] = row[QUESTION_COLUMN]
    payload["schema_name"] = row[schema_column]
    return payload


def post_question(row: pd.Series, thinking: bool) -> Dict[str, Any]:
    """POST one row to /ask and return a normalized result dictionary."""
    payload = build_payload(row, thinking)

    try:
        response = requests.post(
            ASK_URL,
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        result: Dict[str, Any] = {
            "request_payload": payload,
            "http_status_code": response.status_code,
            "request_error": None,
        }

        try:
            response_json = response.json()
        except ValueError:
            response_json = None
            result["raw_response_text"] = response.text

        result["response_json"] = response_json

        if isinstance(response_json, dict):
            result["ask_status"] = response_json.get("status")
            result["ask_answer"] = response_json.get("answer")
            result["ask_reached_end"] = response_json.get("reached_end")
        else:
            result["ask_status"] = None
            result["ask_answer"] = None
            result["ask_reached_end"] = None

        return result

    except requests.RequestException as exc:
        return {
            "request_payload": payload,
            "http_status_code": None,
            "request_error": repr(exc),
            "response_json": None,
            "ask_status": None,
            "ask_answer": None,
            "ask_reached_end": None,
        }

## 6. Optional smoke test with the first row

In [29]:
# Run this cell first to verify your local service is running and the payload shape is correct.
# Comment it out or skip it once verified.

sample_row = df.iloc[0]
print(json.dumps(build_payload(sample_row, True), indent=2, ensure_ascii=False))

sample_result = post_question(sample_row, True)
print(json.dumps(sample_result, indent=2, ensure_ascii=False))

{
  "user_id": "user_123",
  "model": "google/gemma-4-E4B-it",
  "first_user_message": "I want to ask a question about my database schema.",
  "question": "How many singers do we have?",
  "is_thinking": true,
  "skip_user_interaction": true,
  "schema_name": "concert_singer"
}
{
  "request_payload": {
    "user_id": "user_123",
    "model": "google/gemma-4-E4B-it",
    "first_user_message": "I want to ask a question about my database schema.",
    "question": "How many singers do we have?",
    "is_thinking": true,
    "skip_user_interaction": true,
    "schema_name": "concert_singer"
  },
  "http_status_code": 200,
  "request_error": null,
  "response_json": {
    "status": "ok",
    "answer": "SELECT COUNT(singer.singer_id) AS total_singers FROM singer",
    "reached_end": true
  },
  "ask_status": "ok",
  "ask_answer": "SELECT COUNT(singer.singer_id) AS total_singers FROM singer",
  "ask_reached_end": true
}


## 7. Execute requests for every test entry

In [30]:
work_df = df.copy()
if MAX_ROWS is not None:
    work_df = work_df.iloc[OFFSET:OFFSET + MAX_ROWS].copy()

results_thinking = []
results_non_thinking = []

for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="POST /ask"):
    results_thinking.append(post_question(row, True))
    results_non_thinking.append(post_question(row, False))

    if SLEEP_BETWEEN_REQUESTS_SECONDS > 0:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)

# Add both response sets to the original rows with clear column names
work_df["ask_response_thinking"] = results_thinking
work_df["ask_response_non_thinking"] = results_non_thinking

results_df = work_df

print(f"Collected {len(results_df):,} rows with thinking and non-thinking responses")
results_df.head()

POST /ask:   0%|          | 0/234 [00:00<?, ?it/s]

Collected 234 rows with thinking and non-thinking responses


,db_id,query,question,query_toks,query_toks_no_value,question_toks,ask_response_thinking,ask_response_non_thinking
800,world_1,"SELECT Name , SurfaceArea FROM country ORDER ...",What are the names and areas of countries with...,"[SELECT, Name, ,, SurfaceArea, FROM, country, ...","[select, name, ,, surfacearea, from, country, ...","[What, are, the, names, and, areas, of, countr...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
801,world_1,"SELECT Name , SurfaceArea FROM country ORDER ...",Return the names and surface areas of the 5 la...,"[SELECT, Name, ,, SurfaceArea, FROM, country, ...","[select, name, ,, surfacearea, from, country, ...","[Return, the, names, and, surface, areas, of, ...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
802,world_1,SELECT Name FROM country ORDER BY Population D...,What are names of countries with the top 3 lar...,"[SELECT, Name, FROM, country, ORDER, BY, Popul...","[select, name, from, country, order, by, popul...","[What, are, names, of, countries, with, the, t...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
803,world_1,SELECT Name FROM country ORDER BY Population D...,Return the names of the 3 most populated count...,"[SELECT, Name, FROM, country, ORDER, BY, Popul...","[select, name, from, country, order, by, popul...","[Return, the, names, of, the, 3, most, populat...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
804,world_1,SELECT Name FROM country ORDER BY Population A...,What are the names of the nations with the 3 l...,"[SELECT, Name, FROM, country, ORDER, BY, Popul...","[select, name, from, country, order, by, popul...","[What, are, the, names, of, the, nations, with...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."


## 8. Append responses to the original dataframe

In [31]:
# results_df already contains the original Spider rows plus both response sets.
output_df = results_df.copy()

# Useful quick checks.
print("Thinking response counts:")
print(output_df["ask_response_thinking"].value_counts(dropna=False))

print("\nNon-thinking response counts:")
print(output_df["ask_response_non_thinking"].value_counts(dropna=False))

# Error checks, only if these columns exist.
error_columns = [
    col for col in output_df.columns
    if "error" in col.lower()
]

if error_columns:
    print("\nError counts:")
    for col in error_columns:
        print(f"{col}: {output_df[col].notna().sum()}")
else:
    print("\nNo error columns found.")

output_df.head()

Thinking response counts:
ask_response_thinking
{'request_payload': {'user_id': 'user_123', 'model': 'google/gemma-4-E4B-it', 'first_user_message': 'I want to ask a question about my database schema.', 'question': 'What are the names and areas of countries with the top 5 largest area?', 'is_thinking': True, 'skip_user_interaction': True, 'schema_name': 'world_1'}, 'http_status_code': 200, 'request_error': None, 'response_json': {'status': 'ok', 'answer': 'SELECT name, surfacearea FROM country ORDER BY surfacearea DESC LIMIT 5', 'reached_end': True}, 'ask_status': 'ok', 'ask_answer': 'SELECT name, surfacearea FROM country ORDER BY surfacearea DESC LIMIT 5', 'ask_reached_end': True}                                                                                                                                                                                                                                                                                                                       

,db_id,query,question,query_toks,query_toks_no_value,question_toks,ask_response_thinking,ask_response_non_thinking
800,world_1,"SELECT Name , SurfaceArea FROM country ORDER ...",What are the names and areas of countries with...,"[SELECT, Name, ,, SurfaceArea, FROM, country, ...","[select, name, ,, surfacearea, from, country, ...","[What, are, the, names, and, areas, of, countr...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
801,world_1,"SELECT Name , SurfaceArea FROM country ORDER ...",Return the names and surface areas of the 5 la...,"[SELECT, Name, ,, SurfaceArea, FROM, country, ...","[select, name, ,, surfacearea, from, country, ...","[Return, the, names, and, surface, areas, of, ...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
802,world_1,SELECT Name FROM country ORDER BY Population D...,What are names of countries with the top 3 lar...,"[SELECT, Name, FROM, country, ORDER, BY, Popul...","[select, name, from, country, order, by, popul...","[What, are, names, of, countries, with, the, t...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
803,world_1,SELECT Name FROM country ORDER BY Population D...,Return the names of the 3 most populated count...,"[SELECT, Name, FROM, country, ORDER, BY, Popul...","[select, name, from, country, order, by, popul...","[Return, the, names, of, the, 3, most, populat...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."
804,world_1,SELECT Name FROM country ORDER BY Population A...,What are the names of the nations with the 3 l...,"[SELECT, Name, FROM, country, ORDER, BY, Popul...","[select, name, from, country, order, by, popul...","[What, are, the, names, of, the, nations, with...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm..."


## 9. Save enriched results as JSON

In [32]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# orient='records' creates a list of row objects. force_ascii=False preserves non-ASCII text.
output_df.to_json(
    OUTPUT_PATH,
    orient="records",
    indent=2,
    force_ascii=False,
)

print(f"Saved {len(output_df):,} enriched rows to: {OUTPUT_PATH.resolve()}")

Saved 234 enriched rows to: /mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider/notebooks/result/spider_test_with_ask_responses_offset_800_max_rows_234.json


## 10. Inspect saved JSON

In [33]:
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"Loaded {len(saved):,} saved records")
saved[0] if saved else None

Loaded 234 saved records


{'db_id': 'world_1',
 'query': 'SELECT Name ,  SurfaceArea FROM country ORDER BY SurfaceArea DESC LIMIT 5',
 'question': 'What are the names and areas of countries with the top 5 largest area?',
 'query_toks': ['SELECT',
  'Name',
  ',',
  'SurfaceArea',
  'FROM',
  'country',
  'ORDER',
  'BY',
  'SurfaceArea',
  'DESC',
  'LIMIT',
  '5'],
 'query_toks_no_value': ['select',
  'name',
  ',',
  'surfacearea',
  'from',
  'country',
  'order',
  'by',
  'surfacearea',
  'desc',
  'limit',
  'value'],
 'question_toks': ['What',
  'are',
  'the',
  'names',
  'and',
  'areas',
  'of',
  'countries',
  'with',
  'the',
  'top',
  '5',
  'largest',
  'area',
  '?'],
 'ask_response_thinking': {'request_payload': {'user_id': 'user_123',
   'model': 'google/gemma-4-E4B-it',
   'first_user_message': 'I want to ask a question about my database schema.',
   'question': 'What are the names and areas of countries with the top 5 largest area?',
   'is_thinking': True,
   'skip_user_interaction': True